In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as f
from pyspark.sql import types as t
from pyspark.sql.window import Window
from datetime import datetime


## Função Para Ler a Partição

In [0]:
def ler_ultima_particao_delta(spark, base_path):
  """
  Essa fução é para ler a ultima partição dos volumes delta baseada na coluna 'data_processamento'
  """
  try: 
      # Descobrir as partições direto no storage 
      particoes = dbutils.fs.ls(base_path)
      datas = [
              int(p.name.split('=')[1].replace('/', '')) 
              for p in particoes if "data_processamento=" in p.name
      ]
      
      if not datas:
          print(f"Nenhuma partição encontrada em {base_path}")
          return None
      else:
          ultima_particao = max(datas)
          print(f"[{base_path}] Ultima partição: {ultima_particao}")
          return spark.read.format("delta").load(f"{base_path}/data_processamento={ultima_particao}")
  except Exception as e:
    print(f"Erro ao ler caminho {base_path}: {e}")
    return None

## 1. CVM - Informações Diárias 

In [0]:
bronze_path_cvm = "/Volumes/workspace/case_spark_cvm/bronze/cvm_informe_diario/"

df_bronze_cvm = ler_ultima_particao_delta(spark, bronze_path_cvm)

In [0]:
df_bronze_cvm.count()

In [0]:
# Conta a quantidade de valores nulos para cada coluna do dataframe
df_contagem_nulos = df_bronze_cvm.select([
    f.count(f.when(f.col(c).isNull(), c)).alias(c) for c in df_bronze_cvm.columns
])

df_contagem_nulos.show(vertical=True)

### 1.1 tratemento silver

#### 1.1.1 Normalizando CNPJ

In [0]:
df_bronze_cvm = df_bronze_cvm.withColumn(
    "CNPJ_FUNDO_CLASSE",
    f.regexp_replace(f.col("CNPJ_FUNDO_CLASSE"), "[^0-9]", "")
)

df_bronze_cvm = df_bronze_cvm.withColumn(
    "CNPJ_FUNDO_CLASSE",
    f.col("CNPJ_FUNDO_CLASSE").cast("long").cast("string")
)

#### 1.1.2 Retirando dados nulos de Colunas Cores

In [0]:
# Lista de colunas de caso falte dados precisamos dropa 
colunas_obrigatorias = ['TP_FUNDO_CLASSE', 'CNPJ_FUNDO_CLASSE', 'VL_TOTAL']

# Aplicando a filtro para dropar as colunas
df_bronze_cvm = df_bronze_cvm.dropna(subset=colunas_obrigatorias)

#### 1.1.3 Retirando dados duplicados

Com a mudança de rosolução da CVM (***Resolução CVM 175***), Com a nova regra, os fundos passaram a ser estruturados em classes e subclasses, adotando o tipo "CLASSES - FIF" (Fundo de Investimento Financeiro).
Caso acha dados do mesmo ***CNPJ_FUNDO_CLASSE***, os dados de "CLASSES - FIF" terão prioridade e o evento com nomecclatura antiga será excluido.

```
+---------------+------------------+------------+----------+-----------+--------------+-------------+---------+--------+--------+------------------+
|TP_FUNDO_CLASSE| CNPJ_FUNDO_CLASSE|ID_SUBCLASSE| DT_COMPTC|   VL_TOTAL|      VL_QUOTA|VL_PATRIM_LIQ|CAPTC_DIA|RESG_DIA|NR_COTST|data_processamento|
+---------------+------------------+------------+----------+-----------+--------------+-------------+---------+--------+--------+------------------+
|  CLASSES - FIF|12.586.174/0001-67|        NULL|2026-01-07|47445220.02|1.406455790000|  47448983.02|     0.00|    0.00|       1|          20260221|
|             FI|12.586.174/0001-67|        NULL|2026-01-07|47445414.65|1.406474270000|  47449606.58|     0.00|    0.00|       1|          20260221|
+---------------+------------------+------------+----------+-----------+--------------+-------------+---------+--------+--------+------------------+
```

In [0]:
# Coluna temporaria para definir prioridade em CLASSES - FIF
df_bronze_cvm = df_bronze_cvm.withColumn(
    "prioridade_tipo",
    f.when(f.col("TP_FUNDO_CLASSE") ==  "CLASSES - FIF", 1).otherwise(2)
)

# Definindo a janela  particionando pelas colunas CORE
window_spec = Window.partitionBy("CNPJ_FUNDO_CLASSE", "ID_SUBCLASSE", "DT_COMPTC").orderBy("prioridade_tipo")

#Aplicamos a numeração das linhas (row_number) dentro de cada janela
df_bronze_cvm = df_bronze_cvm.withColumn("row_num", f.row_number().over(window_spec))

# filtrando prioridade_tipo = 1 de cada grupo e removendo as colunas auxiliares 
df_bronze_cvm = df_bronze_cvm.filter(f.col("row_num") == 1).drop("prioridade_tipo", "row_num")


#### 1.1.4 Tratamento do Tipo de Dado

In [0]:
df_bronze_cvm = df_bronze_cvm\
    .withColumn('tp_fundo_classe', f.col('TP_FUNDO_CLASSE').cast(t.StringType()))\
    .withColumn('cnpj_fundo_classe', f.col('CNPJ_FUNDO_CLASSE').cast(t.StringType()))\
    .withColumn('id_subclasse', f.col('ID_SUBCLASSE').cast(t.StringType()))\
    .withColumn('dt_comptc', f.col('DT_COMPTC').cast(t.DateType()))\
    .withColumn('vl_total', f.col('VL_TOTAL').cast(t.DecimalType(38,2)))\
    .withColumn('vl_quota', f.col('VL_QUOTA').cast(t.DecimalType(38,11)))\
    .withColumn('vl_patrim_liq', f.col('VL_PATRIM_LIQ').cast(t.DecimalType(38,2)))\
    .withColumn('captc_dia', f.col('CAPTC_DIA').cast(t.DecimalType(38,2)))\
    .withColumn('resg_dia', f.col('RESG_DIA').cast(t.DecimalType(38,2)))\
    .withColumn('nr_cotst', f.col('NR_COTST').cast(t.LongType()))\
    .withColumn('data_processamento', f.col('data_processamento').cast(t.IntegerType()))\
    .drop("TP_FUNDO", "CNPJ_FUNDO")

### 1.2 Salvar na camada Silver

In [0]:
display(df_bronze_cvm.where(f.col("cnpj_fundo_classe") == "8971868000140")) 

In [0]:
data_proc = int(datetime.now().strftime(f"%Y%m%d"))

df_bronze_cvm.write \
    .mode('overwrite') \
    .partitionBy("data_processamento") \
    .format('delta')\
    .option("replaceWhere", f"data_processamento = {data_proc}")\
    .option("mergeSchema", "true") \
    .saveAsTable("workspace.case_spark_cvm.silver_cvm_fundos_diario")

## 2. CVM - Fundos Imobiliarios - Ativo Passivo

In [0]:
bronze_path_ativo_passivo = "/Volumes/workspace/case_spark_cvm/bronze/cvm_fii_ativo_passivo/"

df_bronze_cvm_ativo_passivo = ler_ultima_particao_delta(spark, bronze_path_ativo_passivo)

### 1.1 tratemento silver

#### 1.1.1 Normalizando CNPJ

In [0]:
df_bronze_cvm_ativo_passivo = df_bronze_cvm_ativo_passivo.withColumn(
    "CNPJ_FUNDO_CLASSE",
    f.regexp_replace(f.col("CNPJ_FUNDO_CLASSE"), "[^0-9]", "")
)

df_bronze_cvm_ativo_passivo = df_bronze_cvm_ativo_passivo.withColumn(
    "CNPJ_FUNDO_CLASSE",
    f.col("CNPJ_FUNDO_CLASSE").cast("long").cast("string")
)

#### 1.1.2 Retirando dados nulos de Colunas Cores

In [0]:
# Lista de colunas de caso falte dados precisamos dropa 
colunas_obrigatorias = ['Data_Referencia', 'CNPJ_FUNDO_CLASSE']

# Aplicando a filtro para dropar as colunas
df_bronze_cvm_ativo_passivo = df_bronze_cvm_ativo_passivo.dropna(subset=colunas_obrigatorias)

#### 1.1.3 Tratamento do Tipo de Dado

In [0]:
display(df_bronze_cvm_ativo_passivo)

In [0]:
df_bronze_cvm_ativo_passivo.columns

In [0]:
df_bronze_cvm_ativo_passivo = df_bronze_cvm_ativo_passivo \
    .withColumn('cnpj_fundo_classe', f.col('CNPJ_FUNDO_CLASSE').cast(t.StringType())) \
    .withColumn('data_referencia', f.col('Data_Referencia').cast(t.DateType())) \
    .withColumn('versao', f.col('Versao').cast(t.IntegerType())) \
    .withColumn('total_necessidades_liquidez', f.col('Total_Necessidades_Liquidez').cast(t.DecimalType(22, 2))) \
    .withColumn('disponibilidades', f.col('Disponibilidades').cast(t.DecimalType(22, 2))) \
    .withColumn('titulos_publicos', f.col('Titulos_Publicos').cast(t.DecimalType(22, 2))) \
    .withColumn('titulos_privados', f.col('Titulos_Privados').cast(t.DecimalType(22, 2))) \
    .withColumn('fundos_renda_fixa', f.col('Fundos_Renda_Fixa').cast(t.DecimalType(18, 2))) \
    .withColumn('total_investido', f.col('Total_Investido').cast(t.DecimalType(18, 2))) \
    .withColumn('direitos_bens_imoveis', f.col('Direitos_Bens_Imoveis').cast(t.DecimalType(22, 2))) \
    .withColumn('terrenos', f.col('Terrenos').cast(t.DecimalType(18, 2))) \
    .withColumn('imoveis_renda_acabados', f.col('Imoveis_Renda_Acabados').cast(t.DecimalType(22, 2))) \
    .withColumn('imoveis_renda_construcao', f.col('Imoveis_Renda_Construcao').cast(t.DecimalType(22, 2))) \
    .withColumn('imoveis_venda_acabados', f.col('Imoveis_Venda_Acabados').cast(t.DecimalType(22, 2))) \
    .withColumn('imoveis_venda_construcao', f.col('Imoveis_Venda_Construcao').cast(t.DecimalType(22, 2))) \
    .withColumn('outros_direitos_reais', f.col('Outros_Direitos_Reais').cast(t.DecimalType(22, 2))) \
    .withColumn('acoes', f.col('Acoes').cast(t.DecimalType(22, 2))) \
    .withColumn('debentures', f.col('Debentures').cast(t.DecimalType(22, 2))) \
    .withColumn('bonus_subscricao', f.col('Bonus_Subscricao').cast(t.DecimalType(22, 2))) \
    .withColumn('certificados_deposito_valores_mobiliarios', f.col('Certificados_Deposito_Valores_Mobiliarios').cast(t.DecimalType(22, 2))) \
    .withColumn('cedulas_debentures', f.col('Cedulas_Debentures').cast(t.DecimalType(12, 2))) \
    .withColumn('fundo_acoes', f.col('Fundo_Acoes').cast(t.DecimalType(22, 2))) \
    .withColumn('fip', f.col('FIP').cast(t.DecimalType(22, 2))) \
    .withColumn('fii', f.col('FII').cast(t.DecimalType(22, 2))) \
    .withColumn('fdic', f.col('FDIC').cast(t.DecimalType(22, 2))) \
    .withColumn('outras_cotas_fi', f.col('Outras_Cotas_FI').cast(t.DecimalType(22, 2))) \
    .withColumn('notas_promissorias', f.col('Notas_Promissorias').cast(t.DecimalType(22, 2))) \
    .withColumn('acoes_sociedades_atividades_fii', f.col('Acoes_Sociedades_Atividades_FII').cast(t.DecimalType(22, 2))) \
    .withColumn('cotas_sociedades_atividades_fii', f.col('Cotas_Sociedades_Atividades_FII').cast(t.DecimalType(22, 2))) \
    .withColumn('cepac', f.col('CEPAC').cast(t.DecimalType(22, 2))) \
    .withColumn('cri', f.col('CRI').cast(t.DecimalType(22, 2))) \
    .withColumn('cri_cra', f.col('CRI_CRA').cast(t.DecimalType(22, 2))) \
    .withColumn('letras_hipotecarias', f.col('Letras_Hipotecarias').cast(t.DecimalType(22, 2))) \
    .withColumn('lci', f.col('LCI').cast(t.DecimalType(22, 2))) \
    .withColumn('lci_lca', f.col('LCI_LCA').cast(t.DecimalType(22, 2))) \
    .withColumn('lig', f.col('LIG').cast(t.DecimalType(22, 2))) \
    .withColumn('outros_valores_mobliarios', f.col('Outros_Valores_Mobliarios').cast(t.DecimalType(22, 2))) \
    .withColumn('valores_receber', f.col('Valores_Receber').cast(t.DecimalType(22, 2))) \
    .withColumn('contas_receber_aluguel', f.col('Contas_Receber_Aluguel').cast(t.DecimalType(22, 2))) \
    .withColumn('contas_receber_venda_imoveis', f.col('Contas_Receber_Venda_Imoveis').cast(t.DecimalType(22, 2))) \
    .withColumn('outros_valores_receber', f.col('Outros_Valores_Receber').cast(t.DecimalType(22, 2))) \
    .withColumn('rendimentos_distribuir', f.col('Rendimentos_Distribuir').cast(t.DecimalType(22, 2))) \
    .withColumn('taxa_administracao_pagar', f.col('Taxa_Administracao_Pagar').cast(t.DecimalType(22, 2))) \
    .withColumn('taxa_performance_pagar', f.col('Taxa_Performance_Pagar').cast(t.DecimalType(22, 2))) \
    .withColumn('obrigacoes_aquisicao_imoveis', f.col('Obrigacoes_Aquisicao_Imoveis').cast(t.DecimalType(22, 2))) \
    .withColumn('adiantamento_venda_imoveis', f.col('Adiantamento_Venda_Imoveis').cast(t.DecimalType(22, 2))) \
    .withColumn('adiantamento_alugueis', f.col('Adiantamento_Alugueis').cast(t.DecimalType(22, 2))) \
    .withColumn('obrigacoes_securitizacao_recebiveis', f.col('Obrigacoes_Securitizacao_Recebiveis').cast(t.DecimalType(22, 2))) \
    .withColumn('instrumentos_financeiros_derivativos', f.col('Instrumentos_Financeiros_Derivativos').cast(t.DecimalType(22, 2))) \
    .withColumn('provisoes_contigencias', f.col('Provisoes_Contigencias').cast(t.DecimalType(22, 2))) \
    .withColumn('outros_valores_pagar', f.col('Outros_Valores_Pagar').cast(t.DecimalType(22, 2))) \
    .withColumn('total_passivo', f.col('Total_Passivo').cast(t.DecimalType(22, 2))) \
    .withColumn('data_processamento', f.col('data_processamento').cast(t.IntegerType()))

In [0]:
df_bronze_cvm_ativo_passivo.show()

### 1.2 Salvar na camada Silver

In [0]:
data_proc = int(datetime.now().strftime(f"%Y%m%d"))

df_bronze_cvm_ativo_passivo.write \
    .mode('overwrite')\
    .partitionBy("data_processamento") \
    .format('delta')\
    .option("replaceWhere", f"data_processamento = {data_proc}")\
    .option("mergeSchema", "true") \
    .saveAsTable("workspace.case_spark_cvm.silver_cvm_fii_ativo_passivo")

## 3. CVM - Fundos Imobiliarios - Complemento

In [0]:
bronze_path_complemento = "/Volumes/workspace/case_spark_cvm/bronze/cvm_fii_complemento/"

df_bronze_cvm_complemento = ler_ultima_particao_delta(spark, bronze_path_complemento)

In [0]:
display(df_bronze_cvm_complemento)

In [0]:
display(df_bronze_cvm_complemento.select("percentual_rentabilidade_efetiva_mes").distinct())

In [0]:
display(df_bronze_cvm_complemento.select("percentual_rentabilidade_efetiva_mes").where( f.col("percentual_rentabilidade_efetiva_mes") =='-4E-05'))

### 1.1 tratemento silver

#### 1.1.1 Normalizando CNPJ

In [0]:
df_bronze_cvm_complemento = df_bronze_cvm_complemento.withColumn(
    "CNPJ_FUNDO_CLASSE",
    f.regexp_replace(f.col("CNPJ_FUNDO_CLASSE"), "[^0-9]", "")
)

df_bronze_cvm_complemento = df_bronze_cvm_complemento.withColumn(
    "CNPJ_FUNDO_CLASSE",
    f.col("CNPJ_FUNDO_CLASSE").cast("long").cast("string")
)

#### 1.1.2 Retirando dados nulos de Colunas Cores

In [0]:
# Lista de colunas de caso falte dados precisamos dropa 
colunas_obrigatorias = ['Data_Referencia', 'CNPJ_Fundo_Classe']

# Aplicando a filtro para dropar as colunas
df_bronze_cvm_complemento = df_bronze_cvm_complemento.dropna(subset=colunas_obrigatorias)

#### 1.1.3 Tratamento do Tipo de Dado

In [0]:
df_bronze_cvm_complemento.columns

In [0]:
df_bronze_cvm_complemento = df_bronze_cvm_complemento\
    .withColumn('cnpj_fundo_classe', f.col('CNPJ_FUNDO_CLASSE').cast(t.StringType())) \
    .withColumn('data_referencia', f.col('Data_Referencia').cast(t.DateType())) \
    .withColumn('versao', f.col('Versao').cast(t.IntegerType())) \
    .withColumn('data_informacao_numero_cotistas', f.col('Data_Informacao_Numero_Cotistas').cast(t.DateType())) \
    .withColumn('total_numero_cotistas', f.col('Total_Numero_Cotistas').cast(t.IntegerType())) \
    .withColumn('numero_cotistas_pessoa_fisica', f.col('Numero_Cotistas_Pessoa_Fisica').cast(t.IntegerType())) \
    .withColumn('numero_cotistas_pessoa_juridica_nao_financeira', f.col('Numero_Cotistas_Pessoa_Juridica_Nao_Financeira').cast(t.IntegerType())) \
    .withColumn('numero_cotistas_banco_comercial', f.col('Numero_Cotistas_Banco_Comercial').cast(t.IntegerType())) \
    .withColumn('numero_cotistas_corretora_distribuidora', f.col('Numero_Cotistas_Corretora_Distribuidora').cast(t.IntegerType())) \
    .withColumn('numero_cotistas_outras_pessoas_juridicas_financeira', f.col('Numero_Cotistas_Outras_Pessoas_Juridicas_Financeira').cast(t.IntegerType())) \
    .withColumn('numero_cotistas_investidores_nao_residentes', f.col('Numero_Cotistas_Investidores_Nao_Residentes').cast(t.IntegerType())) \
    .withColumn('numero_cotistas_entidade_aberta_previdencia_complementar', f.col('Numero_Cotistas_Entidade_Aberta_Previdencia_Complementar').cast(t.IntegerType())) \
    .withColumn('numero_cotistas_entidade_fechada_previdencia_complementar', f.col('Numero_Cotistas_Entidade_Fechada_Previd�ncia_Complementar').cast(t.IntegerType())) \
    .withColumn('numero_cotistas_regime_proprio_previdencia_servidores_publicos', f.col('Numero_Cotistas_Regime_Proprio_Previdencia_Servidores_Publicos').cast(t.IntegerType())) \
    .withColumn('numero_cotistas_sociedade_seguradora_resseguradora', f.col('Numero_Cotistas_Sociedade_Seguradora_Resseguradora').cast(t.IntegerType())) \
    .withColumn('numero_cotistas_sociedade_capitalizacao_arrendamento_mercantil', f.col('Numero_Cotistas_Sociedade_Capitalizacao_Arrendamento_Mercantil').cast(t.IntegerType())) \
    .withColumn('numero_cotistas_fii', f.col('Numero_Cotistas_FII').cast(t.IntegerType())) \
    .withColumn('numero_cotistas_outros_fundos', f.col('Numero_Cotistas_Outros_Fundos').cast(t.IntegerType())) \
    .withColumn('numero_cotistas_distribuidores_fundo', f.col('Numero_Cotistas_Distribuidores_Fundo').cast(t.IntegerType())) \
    .withColumn('numero_cotistas_outros_tipos', f.col('Numero_Cotistas_Outros_Tipos').cast(t.IntegerType())) \
    .withColumn('valor_ativo', f.col('Valor_Ativo').cast(t.DecimalType(22, 2))) \
    .withColumn('patrimonio_liquido', f.col('Patrimonio_Liquido').cast(t.DecimalType(22, 2))) \
    .withColumn('cotas_emitidas', f.col('Cotas_Emitidas').cast(t.DecimalType(22, 2))) \
    .withColumn('valor_patrimonial_cotas', f.col('Valor_Patrimonial_Cotas').cast(t.DecimalType(22, 2))) \
    .withColumn('percentual_despesas_taxa_administracao', f.col('Percentual_Despesas_Taxa_Administracao').cast("double")) \
    .withColumn('percentual_despesas_agente_custodiante', f.col('Percentual_Despesas_Agente_Custodiante').cast("double")) \
    .withColumn('percentual_rentabilidade_efetiva_mes', f.col('Percentual_Rentabilidade_Efetiva_Mes').cast("double")) \
    .withColumn('percentual_rentabilidade_patrimonial_mes', f.col('Percentual_Rentabilidade_Patrimonial_Mes').cast("double")) \
    .withColumn('percentual_dividend_yield_mes', f.col('Percentual_Dividend_Yield_Mes').cast("double")) \
    .withColumn('percentual_amortizacao_cotas_mes', f.col('Percentual_Amortizacao_Cotas_Mes').cast("double")) \
    .withColumn('data_processamento', f.col('data_processamento').cast(t.IntegerType()))

In [0]:
df_bronze_cvm_complemento.show()

### 1.2 Salvar na camada Silver

In [0]:
display(df_bronze_cvm_complemento.select("percentual_rentabilidade_efetiva_mes").distinct())

In [0]:
data_proc = int(datetime.now().strftime(f"%Y%m%d"))

df_bronze_cvm_complemento.write \
    .mode('overwrite')\
    .partitionBy("data_processamento") \
    .format('delta')\
    .option("replaceWhere", f"data_processamento = {data_proc}")\
    .option("mergeSchema", "true") \
    .saveAsTable("workspace.case_spark_cvm.silver_cvm_fii_complemento")

## 4. CVM - Fundos Imobiliarios - Geral

In [0]:
bronze_path_geral = "/Volumes/workspace/case_spark_cvm/bronze/cvm_fii_geral/"

df_bronze_cvm_geral = ler_ultima_particao_delta(spark, bronze_path_geral)

In [0]:
display(df_bronze_cvm_geral)

In [0]:
display(df_bronze_cvm_geral.select("CNPJ_Fundo_Classe").distinct().orderBy(f.col("CNPJ_Fundo_Classe").desc()))

In [0]:
for col in ["Data_Referencia", "Data_Entrega", "Data_Funcionamento", "Data_Prazo_Duracao"]:
    print(f"Coluna: {col}")
    df_bronze_cvm_geral.select(col).distinct().show(10, False)

In [0]:
display(df_bronze_cvm_geral.select("Data_Referencia").distinct().orderBy(f.col("data_referencia").desc()))

In [0]:
display(df_bronze_cvm_geral.groupBy("Tipo_Fundo_Classe").count().orderBy(f.col("Tipo_Fundo_Classe").desc()))

### 1.1 tratemento silver

#### 1.1.1 Normalizando CNPJ

In [0]:
df_bronze_cvm_geral = df_bronze_cvm_geral.withColumn(
    "CNPJ_FUNDO_CLASSE",
    f.regexp_replace(f.col("CNPJ_FUNDO_CLASSE"), "[^0-9]", "")
)

df_bronze_cvm_geral = df_bronze_cvm_geral.withColumn(
    "CNPJ_FUNDO_CLASSE",
    f.col("CNPJ_FUNDO_CLASSE").cast("long").cast("string")
)

In [0]:
df_test = spark.read.csv(
    "/Volumes/workspace/case_spark_cvm/raw/cvm_fii/inf_mensal_fii_geral_2026.csv",
    sep=';',
    header=True
)

display(df_test.select("CNPJ_FUNDO_CLASSE"))

In [0]:
display(df_bronze_cvm_geral.orderBy(f.col("Data_Referencia").desc()))

#### 1.1.2 Retirando dados nulos de Colunas Cores

In [0]:
# Lista de colunas de caso falte dados precisamos dropa 
colunas_obrigatorias = ['Data_Referencia', 'CNPJ_FUNDO_CLASSE']

# Aplicando a filtro para dropar as colunas
df_bronze_cvm_ativo_passivo = df_bronze_cvm_ativo_passivo.dropna(subset=colunas_obrigatorias)

#### 1.1.3 Tratamento do Tipo de Dado

In [0]:
display(df_bronze_cvm_geral)

In [0]:
df_bronze_cvm_geral.columns

In [0]:
df_bronze_cvm_geral = df_bronze_cvm_geral\
.withColumn('tipo_fundo_classe', f.col('Tipo_Fundo_Classe').cast(t.StringType())) \
.withColumn('cnpj_fundo_classe', f.col('CNPJ_FUNDO_CLASSE').cast(t.StringType())) \
.withColumn('data_referencia', f.col('Data_Referencia').cast(t.DateType())) \
.withColumn('versao', f.col('Versao').cast(t.IntegerType())) \
.withColumn('data_entrega', f.col('Data_Entrega').cast(t.DateType())) \
.withColumn('nome_fundo_classe', f.col('Nome_Fundo_Classe').cast(t.StringType())) \
.withColumn('data_funcionamento', f.col('Data_Funcionamento').cast(t.DateType())) \
.withColumn('publico_alvo', f.col('Publico_Alvo').cast(t.StringType())) \
.withColumn('codigo_isin', f.col('Codigo_ISIN').cast(t.StringType())) \
.withColumn('quantidade_cotas_emitidas', f.col('Quantidade_Cotas_Emitidas').cast(t.DecimalType(22, 2))) \
.withColumn('fundo_exclusivo', f.col('Fundo_Exclusivo').cast(t.StringType())) \
.withColumn('cotistas_vinculo_familiar', f.col('Cotistas_Vinculo_Familiar').cast(t.StringType())) \
.withColumn('mandato', f.col('Mandato').cast(t.StringType())) \
.withColumn('segmento_atuacao', f.col('Segmento_Atuacao').cast(t.StringType())) \
.withColumn('tipo_gestao', f.col('Tipo_Gestao').cast(t.StringType())) \
.withColumn('prazo_duracao', f.col('Prazo_Duracao').cast(t.StringType())) \
.withColumn('data_prazo_duracao', f.col('Data_Prazo_Duracao').cast(t.DateType())) \
.withColumn('encerramento_exercicio_social', f.col('Encerramento_Exercicio_Social').cast(t.StringType())) \
.withColumn('mercado_negociacao_bolsa', f.col('Mercado_Negociacao_Bolsa').cast(t.StringType())) \
.withColumn('mercado_negociacao_mbo', f.col('Mercado_Negociacao_MBO').cast(t.StringType())) \
.withColumn('mercado_negociacao_mb', f.col('Mercado_Negociacao_MB').cast(t.StringType())) \
.withColumn('entidade_administradora_bvmf', f.col('Entidade_Administradora_BVMF').cast(t.StringType())) \
.withColumn('entidade_administradora_cetip', f.col('Entidade_Administradora_CETIP').cast(t.StringType())) \
.withColumn('nome_administrador', f.col('Nome_Administrador').cast(t.StringType())) \
.withColumn('cnpj_administrador', f.col('CNPJ_Administrador').cast(t.StringType())) \
.withColumn('logradouro', f.col('Logradouro').cast(t.StringType())) \
.withColumn('numero', f.col('Numero').cast(t.StringType())) \
.withColumn('complemento', f.col('Complemento').cast(t.StringType())) \
.withColumn('bairro', f.col('Bairro').cast(t.StringType())) \
.withColumn('cidade', f.col('Cidade').cast(t.StringType())) \
.withColumn('estado', f.col('Estado').cast(t.StringType())) \
.withColumn('cep', f.col('CEP').cast(t.StringType())) \
.withColumn('telefone1', f.col('Telefone1').cast(t.StringType())) \
.withColumn('telefone2', f.col('Telefone2').cast(t.StringType())) \
.withColumn('telefone3', f.col('Telefone3').cast(t.StringType())) \
.withColumn('site', f.col('Site').cast(t.StringType())) \
.withColumn('email', f.col('Email').cast(t.StringType())) \
.withColumn('data_processamento', f.col('data_processamento').cast(t.IntegerType())) 

In [0]:
df_bronze_cvm_geral.show()

### 1.2 Salvar na camada Silver

In [0]:
data_proc = int(datetime.now().strftime(f"%Y%m%d"))

df_bronze_cvm_geral.write \
    .mode('overwrite')\
    .partitionBy("data_processamento") \
    .format('delta')\
    .option("replaceWhere", f"data_processamento = {data_proc}")\
    .option("mergeSchema", "true") \
    .saveAsTable("workspace.case_spark_cvm.silver_cvm_fii_geral")

## 5. registro_classe_cvm


In [0]:
bronze_path_registro_classe_cvm = "/Volumes/workspace/case_spark_cvm/bronze/registro_classe_cvm/"

df_registro_classe_cvm = ler_ultima_particao_delta(spark, bronze_path_registro_classe_cvm)

In [0]:
df_registro_classe_cvm.toPandas()

### 1.1 tratemento silver

#### 1.1.1 Retirando dados nulos de Colunas Cores

In [0]:
# Lista de colunas de caso falte dados precisamos dropa
colunas_obrigatorias = ['ID_Registro_Fundo', 'ID_Registro_Classe', 'CNPJ_Classe', 'Tipo_Classe', 'Data_Inicio','Situacao']

# Aplicando a filtro para dropar as colunas
df_registro_classe_cvm = df_registro_classe_cvm.dropna(subset=colunas_obrigatorias)

#### 1.1.2 Retirando dados duplicados

In [0]:
df_registro_classe_cvm.filter(f.col("CNPJ_Classe") == 32287668000158).toPandas()

Nesta base podem existir registros com ***CNPJ_Classe*** duplicados. Para tratar essa situação, criamos uma coluna de ***prioridade_situacao***, essa coluna irar criar uma regra onde se a situação estiver **"Em Funcionamento Normal"**, vai ter prioridade em seguida mantemos apenas o registro mais recente, considerando o campo ***Data_Registro***.

In [0]:
df_registro_classe_cvm = df_registro_classe_cvm\
    .withColumn("prioridade_situacao", f.when(f.col("Situacao") == "Em Funcionamento Normal", 1).otherwise(2))

window_spec_registros_classe = Window.partitionBy("CNPJ_Classe").orderBy(f.col("prioridade_situacao"), f.col("Data_Registro").desc())

df_registro_classe_cvm = df_registro_classe_cvm.withColumn("row_num", f.row_number().over(window_spec_registros_classe))

df_registro_classe_cvm = df_registro_classe_cvm.filter(f.col("row_num") == 1).drop("row_num", "prioridade_situacao")

#### 1.1.3 Tratamento do Tipo de Dado

In [0]:

df_registro_classe_cvm = df_registro_classe_cvm \
    .withColumn('id_registro_fundo', f.col('ID_Registro_Fundo').cast(t.IntegerType())) \
    .withColumn('id_registro_classe', f.col('ID_Registro_Classe').cast(t.IntegerType())) \
    .withColumn('cnpj_classe', f.col('CNPJ_Classe').cast(t.StringType())) \
    .withColumn('codigo_cvm', f.col('Codigo_CVM').cast(t.IntegerType())) \
    .withColumn('data_registro', f.col('Data_Registro').cast(t.DateType())) \
    .withColumn('data_constituicao', f.col('Data_Constituicao').cast(t.DateType())) \
    .withColumn('data_inicio', f.col('Data_Inicio').cast(t.DateType())) \
    .withColumn('tipo_classe', f.col('Tipo_Classe').cast(t.StringType())) \
    .withColumn('denominacao_social', f.col('Denominacao_Social').cast(t.StringType())) \
    .withColumn('situacao', f.col('Situacao').cast(t.StringType())) \
    .withColumn('data_inicio_situacao', f.col('Data_Inicio_Situacao').cast(t.DateType())) \
    .withColumn('classificacao', f.col('Classificacao').cast(t.StringType())) \
    .withColumn('indicador_desempenho', f.col('Indicador_Desempenho').cast(t.StringType())) \
    .withColumn('classe_cotas', f.col('Classe_Cotas').cast(t.StringType())) \
    .withColumn('classificacao_anbima', f.col('Classificacao_Anbima').cast(t.StringType())) \
    .withColumn('tributacao_longo_prazo', f.col('Tributacao_Longo_Prazo').cast(t.StringType())) \
    .withColumn('entidade_investimento', f.col('Entidade_Investimento').cast(t.StringType())) \
    .withColumn('permitido_aplicacao_cemporcento_exterior', f.col('Permitido_Aplicacao_CemPorCento_Exterior').cast(t.StringType())) \
    .withColumn('classe_esg', f.col('Classe_ESG').cast(t.StringType())) \
    .withColumn('forma_condominio', f.col('Forma_Condominio').cast(t.StringType())) \
    .withColumn('exclusivo', f.col('Exclusivo').cast(t.StringType())) \
    .withColumn('publico_alvo', f.col('Publico_Alvo').cast(t.StringType())) \
    .withColumn('patrimonio_liquido', f.col('Patrimonio_Liquido').cast(t.DecimalType(25,2))) \
    .withColumn('data_patrimonio_liquido', f.col('Data_Patrimonio_Liquido').cast(t.DateType())) \
    .withColumn('cnpj_auditor', f.col('CNPJ_Auditor').cast(t.StringType())) \
    .withColumn('auditor', f.col('Auditor').cast(t.StringType())) \
    .withColumn('cnpj_custodiante', f.col('CNPJ_Custodiante').cast(t.StringType())) \
    .withColumn('custodiante', f.col('Custodiante').cast(t.StringType())) \
    .withColumn('cnpj_controlador', f.col('CNPJ_Controlador').cast(t.StringType())) \
    .withColumn('controlador', f.col('Controlador').cast(t.StringType()))\
    .withColumn('data_processamento', f.col('data_processamento').cast(t.IntegerType()))


In [0]:
display(df_registro_classe_cvm)

### 1.2 Salvar na camada Silver

In [0]:
data_proc = int(datetime.now().strftime(f"%Y%m%d"))

df_registro_classe_cvm.write \
    .mode('overwrite') \
    .partitionBy("data_processamento") \
    .option("replaceWhere", f"data_processamento = {data_proc}")\
    .format('delta')\
    .saveAsTable("workspace.case_spark_cvm.silver_registro_classe_cvm")

## 6. registro_fundo_cvm

In [0]:
bronze_path_registro_fundo_cvm = "/Volumes/workspace/case_spark_cvm/bronze/registro_fundo_cvm/"

df_registro_fundo_cvm = ler_ultima_particao_delta(spark, bronze_path_registro_fundo_cvm)

In [0]:
df_registro_fundo_cvm.toPandas()

### 1.1 tratemento silver

#### 1.1.1 Retirando dados nulos de Colunas Cores

In [0]:
# Lista de colunas de caso falte dados precisamos dropa
colunas_obrigatorias = ['ID_Registro_Fundo', 'CNPJ_Fundo', 'Codigo_CVM', 'Tipo_Fundo', 'Situacao']

# Aplicando a filtro para dropar as colunas
df_registro_fundo_cvm = df_registro_fundo_cvm.dropna(subset=colunas_obrigatorias)

#### 1.1.2 Retirando dados duplicados

Nesta base podem existir registros com ***CNPJ_Fundo*** duplicados. Para tratar essa situação, criamos uma coluna de ***prioridade_situacao***, essa coluna irar criar uma regra onde se a situação estiver **"Em Funcionamento Normal"**, vai ter prioridade em seguida mantemos apenas o registro mais recente, considerando o campo ***Data_Registro***.

In [0]:
df_registro_fundo_cvm = df_registro_fundo_cvm\
    .withColumn("prioridade_situacao", f.when(f.col("Situacao") == "Em Funcionamento Normal", 1).otherwise(2))

window_spec_registros_classe = Window.partitionBy("CNPJ_Fundo").orderBy(f.col("prioridade_situacao"), f.col("Data_Registro").desc())

df_registro_fundo_cvm = df_registro_fundo_cvm.withColumn("row_num", f.row_number().over(window_spec_registros_classe))

df_registro_fundo_cvm = df_registro_fundo_cvm.filter(f.col("row_num") == 1).drop("row_num", "prioridade_situacao")

#### 1.1.3 Tratamento do Tipo de Dado

In [0]:
df_registro_fundo_cvm = df_registro_fundo_cvm \
    .withColumn('id_registro_fundo', f.col('ID_Registro_Fundo').cast(t.IntegerType())) \
    .withColumn('cnpj_fundo', f.col('CNPJ_Fundo').cast(t.StringType())) \
    .withColumn('codigo_cvm', f.col('Codigo_CVM').cast(t.IntegerType())) \
    .withColumn('data_registro', f.col('Data_Registro').cast(t.DateType())) \
    .withColumn('data_constituicao', f.col('Data_Constituicao').cast(t.DateType())) \
    .withColumn('tipo_fundo', f.col('Tipo_Fundo').cast(t.StringType())) \
    .withColumn('denominacao_social', f.col('Denominacao_Social').cast(t.StringType())) \
    .withColumn('data_cancelamento', f.col('Data_Cancelamento').cast(t.DateType())) \
    .withColumn('situacao', f.col('Situacao').cast(t.StringType())) \
    .withColumn('data_inicio_situacao', f.col('Data_Inicio_Situacao').cast(t.DateType())) \
    .withColumn('data_adaptacao_rcvm175', f.col('Data_Adaptacao_RCVM175').cast(t.DateType())) \
    .withColumn('data_inicio_exercicio_social', f.col('Data_Inicio_Exercicio_Social').cast(t.DateType())) \
    .withColumn('data_fim_exercicio_social', f.col('Data_Fim_Exercicio_Social').cast(t.DateType())) \
    .withColumn('patrimonio_liquido', f.col('Patrimonio_Liquido').cast(t.DecimalType(25,2))) \
    .withColumn('data_patrimonio_liquido', f.col('Data_Patrimonio_Liquido').cast(t.DateType())) \
    .withColumn('diretor', f.col('Diretor').cast(t.StringType())) \
    .withColumn('cnpj_administrador', f.col('CNPJ_Administrador').cast(t.StringType())) \
    .withColumn('administrador', f.col('Administrador').cast(t.StringType())) \
    .withColumn('tipo_pessoa_gestor', f.col('Tipo_Pessoa_Gestor').cast(t.StringType())) \
    .withColumn('cpf_cnpj_gestor', f.col('CPF_CNPJ_Gestor').cast(t.StringType())) \
    .withColumn('gestor', f.col('Gestor').cast(t.StringType())) \
    .withColumn('data_processamento', f.col('data_processamento').cast(t.IntegerType()))


### 1.2 Salvar na camada Silver

In [0]:
data_proc = int(datetime.now().strftime(f"%Y%m%d"))

df_registro_fundo_cvm.write \
    .mode('overwrite') \
    .partitionBy("data_processamento") \
    .option("replaceWhere", f"data_processamento = {data_proc}")\
    .format('delta')\
    .saveAsTable("workspace.case_spark_cvm.silver_registro_fundo_cvm")

## 7. registro_subclasse_cvm

In [0]:
bronze_path_registro_subclasse_cvm = "/Volumes/workspace/case_spark_cvm/bronze/registro_subclasse_cvm/"

df_registro_subclasse_cvm = ler_ultima_particao_delta(spark, bronze_path_registro_subclasse_cvm)

### 1.1 tratemento silver

#### 1.1.1 Retirando dados nulos de Colunas Cores

In [0]:
# Lista de colunas de caso falte dados precisamos dropala 
colunas_obrigatorias = ['ID_Registro_Classe', 'ID_Subclasse', 'Situacao']

# Aplicando a filtro para dropar as colunas
df_registro_subclasse_cvm = df_registro_subclasse_cvm.dropna(subset=colunas_obrigatorias)

#### 1.1.2 Retirando dados duplicados

Em **registro_subclasse** só retiramos os fundos com ***Situacao*** de **Cancelado**

In [0]:
df_registro_subclasse_cvm = df_registro_subclasse_cvm.filter(f.col("Situacao") != 'Cancelado')

#### 1.1.3 Tratamento do Tipo de Dado

In [0]:
df_registro_subclasse_cvm = df_registro_subclasse_cvm \
    .withColumn('id_registro_classe', f.col('ID_Registro_Classe').cast(t.IntegerType())) \
    .withColumn('id_subclasse', f.col('ID_Subclasse').cast(t.StringType())) \
    .withColumn('codigo_cvm', f.col('Codigo_CVM').cast(t.IntegerType())) \
    .withColumn('data_constituicao', f.col('Data_Constituicao').cast(t.DateType())) \
    .withColumn('data_inicio', f.col('Data_Inicio').cast(t.DateType())) \
    .withColumn('denominacao_social', f.col('Denominacao_Social').cast(t.StringType())) \
    .withColumn('situacao', f.col('Situacao').cast(t.StringType())) \
    .withColumn('data_inicio_situacao', f.col('Data_Inicio_Situacao').cast(t.DateType())) \
    .withColumn('forma_condominio', f.col('Forma_Condominio').cast(t.StringType())) \
    .withColumn('exclusivo', f.col('Exclusivo').cast(t.StringType())) \
    .withColumn('publico_alvo', f.col('Publico_Alvo').cast(t.StringType())) \
    .withColumn('previdenciario', f.col('Previdenciario').cast(t.StringType())) \
    .withColumn('exclusivo_inr', f.col('Exclusivo_INR').cast(t.StringType())) \
    .withColumn('exclusivo_previdencia_complementar', f.col('Exclusivo_Previdencia_Complementar').cast(t.StringType())) \
    .withColumn('data_processamento', f.col('data_processamento').cast(t.IntegerType()))


### 1.2 Salvar na camada Silver

In [0]:
data_proc = int(datetime.now().strftime(f"%Y%m%d"))

df_registro_subclasse_cvm.write \
    .mode('overwrite') \
    .partitionBy("data_processamento") \
    .option("replaceWhere", f"data_processamento = {data_proc}")\
    .format('delta')\
    .saveAsTable("workspace.case_spark_cvm.silver_registro_subclasse_cvm")

## 8. Valores Indicador Desempenho

### 1.1 tratemento silver

#### 1.1.1 LEITURA DOS DADOS BRONZE

In [0]:
bronze_path_selic = "/Volumes/workspace/case_spark_cvm/bronze/data_selic/"
df_bronze_selic = ler_ultima_particao_delta(spark, bronze_path_selic)

bronze_path_cdi = "/Volumes/workspace/case_spark_cvm/bronze/data_cdi_diario/"
df_bronze_cdi = ler_ultima_particao_delta(spark, bronze_path_cdi)

bronze_path_ipca = "/Volumes/workspace/case_spark_cvm/bronze/data_ipca_mensal/"
df_bronze_ipca = ler_ultima_particao_delta(spark, bronze_path_ipca)

bronze_path_ibov = "/Volumes/workspace/case_spark_cvm/bronze/data_ibov/"
df_bronze_ibov = ler_ultima_particao_delta(spark, bronze_path_ibov)

#### 1.1.2 TRATAMENTO SELIC, CDI E IBOV

In [0]:
df_selic = df_bronze_selic\
    .withColumn(
    "data",
    f.date_format(f.to_date(f.col("data"), "dd/MM/yyy"), "yyyy-MM-dd")
    )\
    .withColumn("data", f.col("data").cast(t.DateType()))\
    .withColumn("valor", f.col("valor").cast(t.DecimalType(10,2)))\
    .withColumn("data_processamento", f.col("data_processamento").cast(t.IntegerType()))\
    .withColumnRenamed("valor", "valor_selic")


df_cdi = df_bronze_cdi\
    .withColumn(
    "data",
    f.date_format(f.to_date(f.col("data"), "dd/MM/yyy"), "yyyy-MM-dd")
    )\
    .withColumn("data", f.col("data").cast(t.DateType()))\
    .withColumn("valor", f.col("valor").cast(t.DecimalType(10,6)))\
    .withColumn("data_processamento", f.col("data_processamento").cast(t.IntegerType()))\
    .withColumnRenamed("valor", "valor_cdi")


df_ibov = df_bronze_ibov\
    .withColumn(
    "data",
        f.from_unixtime(f.col("timestamp")).cast("date")
    )\
    .withColumn("data", f.col("data").cast(t.DateType()))\
    .withColumn("close", f.col("close").cast(t.DecimalType(20,6)))\
    .withColumn("data_processamento", f.col("data_processamento").cast(t.IntegerType()))\
    .withColumnRenamed("close", "ibov_close")\
    .drop("timestamp")\
    .select("data", "ibov_close", "data_processamento")

#### 1.1.3 TRATAMENTO IPCA (MENSAL) E CÁLCULO DO ACUMULADO (12 MESES)

In [0]:
df_bronze_ipca = df_bronze_ipca\
    .withColumn("data", f.date_format(f.to_date(f.col("data"), "dd/MM/yyy"), "yyyy-MM-dd"))\
    .withColumn("data", f.col("data").cast(t.DateType()))\
    .withColumn("valor", f.col("valor").cast(t.DecimalType(10,4)))\
    .withColumn("data_processamento", f.col("data_processamento").cast(t.IntegerType()))\
    .withColumnRenamed("valor", "ipca_mensal")\
    .withColumnRenamed("data", "data_ipca")
    
# Para calcular o IPCA acumulado de 12 meses corretamente (juros compostos)
# Fator = 1 + (ipca_mensal / 100)
df_ipca = df_bronze_ipca.withColumn("fator", (f.col("ipca_mensal") / 100) + 1)

# Usamos uma Window para pegar os últimos 12 meses ordenados pela data
# Acumulado = (Produto dos fatores de 12 meses) - 1. No PySpark: EXP(SUM(LOG(fator)))
window_12m = Window.orderBy("data_ipca").rowsBetween(-11, Window.currentRow)

df_ipca = df_ipca \
    .withColumn("fator_acumulado", f.exp(f.sum(f.log("fator")).over(window_12m))) \
    .withColumn("ipca_anual", ((f.col("fator_acumulado") - 1) * 100).cast(t.DecimalType(10, 2)))

# Criamos uma chave Ano-Mês para facilitar o join com os dados diários
df_ipca = df_ipca \
    .withColumn("ano_mes", f.date_format("data_ipca", "yyyy-MM")) \
    .select("ano_mes", "ipca_mensal", "ipca_anual")

#### 1.1.4 CRIAÇÃO DE UM CALENDÁRIO ÚNICO E JOIN DOS INDICADORES


In [0]:
# Extraímos todas as datas únicas disponíveis entre Selic e CDI

df_datas = df_selic.select("data").union(df_cdi.select("data")).distinct()

# Criamos a chave Ano-Mês nas datas base
df_datas = df_datas.withColumn("ano_mes", f.date_format("data", "yyyy-MM"))

# Realizamos o Join: left com Selic, left com CDI, left com IPCA
df_indicadores = df_datas \
    .join(df_selic, "data", "left") \
    .join(df_cdi, "data", "left") \
    .join(df_ibov, "data", "left")\
    .join(df_ipca, "ano_mes", "left")




#### 1.1.5 CONTORNO DO PROBLEMA DE IPCA ATRASADO (FORWARD FILL)

In [0]:
# Para os dias cujos meses ainda não têm IPCA lançado (Ex: fev/mar de 2026 ficarão nulos no join),
# preenchemos com o último valor de IPCA conhecido usando a função last() com ignorenulls=True.
window_ffill = Window.orderBy("data").rowsBetween(Window.unboundedPreceding, Window.currentRow)

df_indicadores = df_indicadores \
    .withColumn("ipca_mensal", f.last("ipca_mensal", ignorenulls=True).over(window_ffill)) \
    .withColumn("ipca_anual", f.last("ipca_anual", ignorenulls=True).over(window_ffill))


# Limpamos a tabela para o formato final e adicionamos data_processamento
data_proc = int(datetime.now().strftime("%Y%m%d"))

df_silver_indicadores = df_indicadores \
    .select("data", "valor_selic", "valor_cdi", "ipca_mensal", "ipca_anual", "ibov_close") \
    .withColumn("data_processamento", f.lit(data_proc).cast(t.IntegerType()))

In [0]:
display(df_silver_indicadores.orderBy(f.col("data").desc()))

### 1.2 Salvar na camada Silver

In [0]:

df_silver_indicadores.write \
    .mode('overwrite') \
    .partitionBy("data_processamento") \
    .option("replaceWhere", f"data_processamento = {data_proc}")\
    .format('delta')\
    .saveAsTable("workspace.case_spark_cvm.silver_dados_indicadores_economicos")

In [0]:
df_silver_indicadores.orderBy(f.col("data").desc()).show()